In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
from geocutool.primitive.gs import compute_gaussian_aabb

# --- 1. Your existing PyTorch setup ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

means = torch.tensor([[0.0, 0.0, 0.0]]).to(device)
rotations = torch.tensor([[0.5, 0.2, 0.7, 0.3]]).to(device)
scales = torch.tensor([[0.1, 0.6, 0.5]]).to(device)

level = 8
iso = 11.345
tol = 1. / 8.
rotnorm = True

# Compute AABB from your C++ backend
aabb_min, aabb_max = compute_gaussian_aabb(means, rotations, scales, iso=iso, tol=tol, level=level, rotnorm=rotnorm)

print("AABB Min:", aabb_min)
print("AABB Max:", aabb_max)

# --- 2. Extract Data to CPU/Numpy for Plotly ---
# We only plot the first Gaussian (index 0)
mean_np = means[0].cpu().numpy()
rot_np = rotations[0].cpu().numpy()
scale_np = scales[0].cpu().numpy()

min_pt = aabb_min[0].cpu().numpy()
max_pt = aabb_max[0].cpu().numpy()

# Replicate the C++ scaling limit math
voxelSize = 2.0 / (1 << level)
min_scale = tol * voxelSize
scale_np = np.maximum(scale_np, min_scale)

# --- 3. Generate the Gaussian Ellipsoid ---
# Create a base parametric sphere
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

# Scale the sphere using your iso threshold
# Extents = scale * sqrt(iso)
rx, ry, rz = scale_np * np.sqrt(iso)
x_ellipsoid = x_sphere * rx
y_ellipsoid = y_sphere * ry
z_ellipsoid = z_sphere * rz

# Flatten for matrix multiplication
points = np.vstack([x_ellipsoid.flatten(), y_ellipsoid.flatten(), z_ellipsoid.flatten()])

# Replicate your exact C++ Quaternion-to-Rotation-Matrix math
r, x, y, z = rot_np[0], rot_np[1], rot_np[2], rot_np[3]
if rotnorm:
    inv_norm = 1.0 / np.sqrt(r*r + x*x + y*y + z*z)
    r *= inv_norm; x *= inv_norm; y *= inv_norm; z *= inv_norm

R_mat = np.array([
    [1 - 2*(y*y + z*z), 2*(x*y - r*z), 2*(x*z + r*y)],
    [2*(x*y + r*z), 1 - 2*(x*x + z*z), 2*(y*z - r*x)],
    [2*(x*z - r*y), 2*(y*z + r*x), 1 - 2*(x*x + y*y)]
])

# Apply Rotation and Translation
rotated_points = R_mat @ points
x_final = rotated_points[0, :].reshape(x_sphere.shape) + mean_np[0]
y_final = rotated_points[1, :].reshape(y_sphere.shape) + mean_np[1]
z_final = rotated_points[2, :].reshape(z_sphere.shape) + mean_np[2]

# --- 4. Generate the AABB Wireframe ---
# Define the 8 corners of the bounding box
xc = [min_pt[0], max_pt[0], max_pt[0], min_pt[0], min_pt[0], max_pt[0], max_pt[0], min_pt[0]]
yc = [min_pt[1], min_pt[1], max_pt[1], max_pt[1], min_pt[1], min_pt[1], max_pt[1], max_pt[1]]
zc = [min_pt[2], min_pt[2], min_pt[2], min_pt[2], max_pt[2], max_pt[2], max_pt[2], max_pt[2]]

# Create a continuous line path that draws the 12 edges of the box
# "None" tells Plotly to lift the pen and break the line
x_lines = [xc[0], xc[1], xc[2], xc[3], xc[0], None, xc[4], xc[5], xc[6], xc[7], xc[4], None, xc[0], xc[4], None, xc[1], xc[5], None, xc[2], xc[6], None, xc[3], xc[7]]
y_lines = [yc[0], yc[1], yc[2], yc[3], yc[0], None, yc[4], yc[5], yc[6], yc[7], yc[4], None, yc[0], yc[4], None, yc[1], yc[5], None, yc[2], yc[6], None, yc[3], yc[7]]
z_lines = [zc[0], zc[1], zc[2], zc[3], zc[0], None, zc[4], zc[5], zc[6], zc[7], zc[4], None, zc[0], zc[4], None, zc[1], zc[5], None, zc[2], zc[6], None, zc[3], zc[7]]

# --- 5. Render with Plotly ---
fig = go.Figure()

# Add the Gaussian Ellipsoid
fig.add_trace(go.Surface(
    x=x_final, y=y_final, z=z_final, 
    opacity=0.6, 
    colorscale='Plasma', 
    showscale=False,
    name='Gaussian'
))

# Add the AABB Wireframe
fig.add_trace(go.Scatter3d(
    x=x_lines, y=y_lines, z=z_lines,
    mode='lines',
    line=dict(color='red', width=4),
    name='AABB'
))

# Force the 3D axes to have equal scale so the box isn't distorted
fig.update_layout(
    scene=dict(aspectmode='data'),
    scene_camera=dict(projection=dict(type="orthographic")),
    title="3D Gaussian and Computed AABB",
    # margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

AABB Min: tensor([[-1.5920, -1.4965, -1.5033]], device='cuda:0')
AABB Max: tensor([[1.5920, 1.4965, 1.5033]], device='cuda:0')
